In [1]:
!pip -q install transformers datasets accelerate torchaudio librosa soundfile sentencepiece

In [4]:
import os
import torch
from transformers import pipeline
from google.colab import files

In [5]:
device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device
)

Device: CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

In [10]:
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print("Uploaded file:", audio_file)

Saving Recording.m4a to Recording.m4a
Uploaded file: Recording.m4a


In [14]:
!apt-get -qq install ffmpeg

import os

wav_file = "converted_audio.wav"
!ffmpeg -i "$audio_file" -ar 16000 -ac 1 "$wav_file" -y

print("Converted to:", wav_file)
print("File exists:", os.path.exists(wav_file))

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [17]:
!pip -q install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.0 MB/s eta 0:00:00


In [18]:
import whisper

model = whisper.load_model("small")

100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 68.9MiB/s]


In [19]:
result = model.transcribe("converted_audio.wav")
transcript = result["text"]

print("TRANSCRIPT:")
print(transcript)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


TRANSCRIPT:
 Today we reviewed the progress of the project and discussed the next steps. We finished the first part of the system and tested the initial model output. We still need to improve the evaluation section, update the slides and upload the final files to GitHub. By tomorrow we should complete the demo video, review the results and prepare for the final submission.


In [20]:
def generate_summary(text):
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    short_summary = ". ".join(sentences[:3])
    if short_summary and not short_summary.endswith("."):
        short_summary += "."
    return short_summary

def generate_action_items(text):
    action_keywords = [
        "need to", "should", "must", "follow up", "complete", "submit",
        "send", "finish", "prepare", "review", "update", "upload", "improve"
    ]
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    actions = [s for s in sentences if any(k in s.lower() for k in action_keywords)]
    return actions[:5]

def extract_keywords(text):
    words = text.lower().replace("\n", " ").split()
    stopwords = {
        "the","a","an","and","or","but","is","are","was","were","to","of","in","on","for",
        "with","we","you","they","it","this","that","be","as","at","by","from","our","their",
        "will","would","should","can","could","have","has","had","do","does","did","need"
    }
    cleaned = [w.strip(".,!?;:()[]{}\"'") for w in words]
    cleaned = [w for w in cleaned if w and w not in stopwords and len(w) > 3]

    freq = {}
    for w in cleaned:
        freq[w] = freq.get(w, 0) + 1

    keywords = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:8]
    return [k for k, _ in keywords]

summary = generate_summary(transcript)
action_items = generate_action_items(transcript)
keywords = extract_keywords(transcript)

print("SUMMARY:")
print(summary)

print("\nACTION ITEMS:")
for i, item in enumerate(action_items, 1):
    print(f"{i}. {item}")

print("\nKEYWORDS:")
print(keywords)

SUMMARY:
Today we reviewed the progress of the project and discussed the next steps. We finished the first part of the system and tested the initial model output. We still need to improve the evaluation section, update the slides and upload the final files to GitHub.

ACTION ITEMS:
1. Today we reviewed the progress of the project and discussed the next steps
2. We finished the first part of the system and tested the initial model output
3. We still need to improve the evaluation section, update the slides and upload the final files to GitHub
4. By tomorrow we should complete the demo video, review the results and prepare for the final submission

KEYWORDS:
['final', 'today', 'reviewed', 'progress', 'project', 'discussed', 'next', 'steps']


In [21]:
baseline_output = transcript

improved_output = {
    "transcript": transcript,
    "summary": summary,
    "action_items": action_items,
    "keywords": keywords
}

print("BASELINE OUTPUT:\n")
print(baseline_output)

print("\n" + "="*60 + "\n")

print("IMPROVED OUTPUT:\n")
print("Transcript:", improved_output["transcript"])
print("\nSummary:", improved_output["summary"])
print("\nAction Items:", improved_output["action_items"])
print("\nKeywords:", improved_output["keywords"])

BASELINE OUTPUT:

 Today we reviewed the progress of the project and discussed the next steps. We finished the first part of the system and tested the initial model output. We still need to improve the evaluation section, update the slides and upload the final files to GitHub. By tomorrow we should complete the demo video, review the results and prepare for the final submission.


IMPROVED OUTPUT:

Transcript:  Today we reviewed the progress of the project and discussed the next steps. We finished the first part of the system and tested the initial model output. We still need to improve the evaluation section, update the slides and upload the final files to GitHub. By tomorrow we should complete the demo video, review the results and prepare for the final submission.

Summary: Today we reviewed the progress of the project and discussed the next steps. We finished the first part of the system and tested the initial model output. We still need to improve the evaluation section, update th

In [22]:
import os

os.makedirs("outputs", exist_ok=True)

with open("outputs/transcript.txt", "w") as f:
    f.write(transcript)

with open("outputs/summary.txt", "w") as f:
    f.write(summary)

with open("outputs/action_items.txt", "w") as f:
    for i, item in enumerate(action_items, 1):
        f.write(f"{i}. {item}\n")

with open("outputs/keywords.txt", "w") as f:
    f.write(", ".join(keywords))

print("Saved outputs to /outputs")

Saved outputs to /outputs
